In [60]:
from pypdf import PdfReader
path = "docs/nipun_david_linkedin.pdf"
reader = PdfReader(path)

In [61]:
resume = ""
for page in reader.pages:
    resume += page.extract_text()

print(resume[:100])

   
Contact
nipun18david@gmail.com
www.linkedin.com/in/nipundavid
(LinkedIn)
Top Skills
Multi-agent 


In [96]:
system_prompt = f"""
You are a helpful assistant who has all the information from the {resume} about the person which is from Linkedin profile and 
whenever a questions user asks you a question about the person, you will answer it based on the information. 
If the information is not present/available, you will inform the user that the information is not publicly available, hence you can send
an email, so share your email id and I will send an email to the person on your behalf.
"""

In [97]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def send_email(email: str, body:str) -> str:
    """Send email of the user to the recipient 
    who in this case is the person whose linkedin information is being read 
    with the body of the email."""
    return f"Email sent to {email} with body: {body}!"

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    tools=[send_email],
    system_prompt=system_prompt,
)

In [98]:
# A quick test
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Tell me about Nipun David in 100 words?"}]}
)

In [99]:
messages = result.get("messages", [])
if messages:
    print(messages[-1].content[0]["text"])
else:
    print(result)

Nipun David is a highly experienced software engineer with over 15 years in software development and 6 years in consulting. Currently serving as a Software Engineer Senior Manager for Gen AI/LLM at BCG, he specializes in building enterprise-grade, agentic AI systems and LLMOps foundations. Previously, at Nagarro, he led significant RAG and GenAI pipeline deployments. His expertise spans LangGraph, cloud-native applications, and AR/VR platforms. Nipun holds a Master of Technology from BITS Pilani and a Postgraduate Certificate in AI & ML from UT Austin, reflecting his commitment to driving innovation through scalable, production-ready AI solutions.


In [100]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "I need to get in touch with Nipun David, "
    "can you send an email to him on my behalf? requesting his mobile number. My Email id is test@example.com"
    }]}
)

In [101]:
messages = result.get("messages", [])
if messages:
    print(messages[-1].content[0]["text"])
else:
    print(result)

The email requesting Nipun David's mobile number has been sent to nipun18david@gmail.com on your behalf.


In [102]:
messages = result.get("messages", [])

for msg in messages:
    if getattr(msg, "tool_calls", None):
        tool_call = msg.tool_calls[0]
        print("Tool name:", tool_call["name"])
        print("Tool args:", tool_call["args"])

        tool_result = send_email.invoke(tool_call["args"])
        print("Tool result:", tool_result)
        break

Tool name: send_email
Tool args: {'email': 'nipun18david@gmail.com', 'body': 'Hello Nipun David,\n\nI am writing to you on behalf of a user (test@example.com) who would like to get in touch with you. They have requested your mobile number.\n\nBest regards,\nYour AI Assistant'}
Tool result: Email sent to nipun18david@gmail.com with body: Hello Nipun David,

I am writing to you on behalf of a user (test@example.com) who would like to get in touch with you. They have requested your mobile number.

Best regards,
Your AI Assistant!


## Wrapping up everything in Gradio Interface

In [103]:
import gradio as gr

def greet(message: str, history) -> str:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": message}]}
    )
    messages = result.get("messages", [])
    if messages:
        last_message = messages[-1]
        content = getattr(last_message, "content", "")
        if isinstance(content, list):
            texts = []
            for item in content:
                if isinstance(item, dict):
                    texts.append(item.get("text", ""))
                else:
                    texts.append(str(item))
            return "\n".join(texts)
        return str(content)
    return str(result)

demo = gr.ChatInterface(fn=greet, 
                        description="Ask me anything about Nipun David's resume and I will answer based on the information from the [LinkedIn](https://www.linkedin.com/in/nipundavid/) profile.",
                        autoscroll=True,
                        title="Digital Nipun David")
demo.launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.
